# sweets example 3 — NISAR GSLC workflow

End-to-end InSAR run over the same LA AOI, this time using pre-geocoded [NISAR L2 GSLC HDF5s](https://nisar.jpl.nasa.gov/) pulled from CMR. NISAR ships UTM-projected complex imagery at L-band, so the pipeline is the simplest of the three sources: no COMPASS, no burst database, no orbit files, no geometry stitching. sweets writes a ~1 KB VRT wrapper per polarization that injects the correct geotransform on top of the HDF5 subdataset (GDAL's HDF5 driver can't parse NISAR's separate xCoordinates/yCoordinates arrays), and hands the VRTs directly to dolphin.

**Why use it?** NISAR is L-band (~24 cm wavelength) — cuts through vegetation and decorrelates much more slowly than Sentinel-1 C-band, so coherent stacks can span months instead of weeks.

**AOI / time**: same LA polygon + 2025-12-01 -> 2025-12-31 window as the Sentinel-1 and OPERA example notebooks. NISAR track 34 frame 18 (the ASF Vertex fields `Track` and `Frame`) covers LA on an ascending pass during this period.

**Expected wall time**: ~30 s to a couple of minutes, dominated by the HDF5 subset download. dolphin itself takes about 15 seconds on this small stack.

## Earthdata credentials

sweets downloads Sentinel-1 bursts, OPERA CSLCs, NISAR GSLCs and tropo data from NASA Earthdata endpoints. Every subsystem used below (`burst2safe`, `opera-utils`, `asf_search`) expects a `~/.netrc` entry like:

```
machine urs.earthdata.nasa.gov
  login YOUR_EARTHDATA_USERNAME
  password YOUR_EARTHDATA_PASSWORD
```

If you don't have credentials yet, register for free at <https://urs.earthdata.nasa.gov/users/new>.

## Configure the run

`--source nisar-gslc` switches to the NISAR path. `--track` and `--frame` together pin a specific NISAR repeat-pass stack (they're the `Track` and `Frame` fields on ASF Vertex, and also the `RRR` / `TTT` digits in the granule filename). `--frequency A` asks for the primary L-band channel; `--polarizations HH` picks the single-polarization channel that every NISAR BETA PR product ships.

Tropo correction is not yet wired for NISAR (no stitched local_incidence_angle raster on this path), so `--do-tropo` is not passed.

In [ ]:
from pathlib import Path

WORK_DIR = Path("./example_nisar").resolve()
WORK_DIR.mkdir(exist_ok=True)
CONFIG_YAML = WORK_DIR / "sweets_config.yaml"
print(WORK_DIR)

In [ ]:
!sweets config \
  --bbox=-118.3581 33.7005 -118.2128 33.8316 \
  --start 2025-12-01 --end 2025-12-31 \
  --source nisar-gslc \
  --track 34 --frame 18 \
  --frequency A \
  --polarizations HH \
  --out-dir {WORK_DIR}/data \
  --work-dir {WORK_DIR} \
  --output {CONFIG_YAML}

In [ ]:
!head -60 {CONFIG_YAML}

## Run the workflow

Under the hood:

1. sweets queries CMR for NISAR GSLC products on track 34 frame 18 that intersect the AOI, ranks the returned granules by (stack size, polarization match, frequency match), and picks the largest coherent signature group.
2. For each product in that group, sweets asks opera-utils for a bbox-subset of the HDF5 and verifies it actually has data in the AOI (NISAR PR products can advertise a frequency whose actual grid extent is narrower than the bounding polygon — those get skipped automatically and sweets falls through to the next signature).
3. Each successful subset gets a per-polarization VRT wrapper. dolphin opens those as normal rasters.
4. dolphin runs phase linking, interferogram network selection, SNAPHU unwrapping, timeseries inversion and velocity estimation as usual. The NISAR L-band wavelength is auto-detected from the HDF5 MODE code so outputs land in meters, not radians.

In [ ]:
!sweets run {CONFIG_YAML}

## Inspect the outputs

In [ ]:
!ls {WORK_DIR}/dolphin/

In [ ]:
!ls {WORK_DIR}/dolphin/timeseries/

In [ ]:
# Velocity raster: NISAR L-band, ~24 cm wavelength.
!gdalinfo -stats {WORK_DIR}/dolphin/timeseries/velocity.tif | grep -E 'Size|Pixel Size|Unit Type|Minimum|Maximum|StdDev'

## Under-the-hood: the VRT wrappers

The `data/` directory holds both the raw HDF5 subset and tiny VRT wrappers:

In [ ]:
!ls -lh {WORK_DIR}/data/ 2>&1 | head -10

In [ ]:
# Peek at one VRT to see how sweets injects the geotransform over
# the HDF5 subdataset.
import glob
vrts = sorted(glob.glob(f"{WORK_DIR}/data/*.vrt"))
if vrts:
    print(open(vrts[0]).read())